In [29]:
#### Core script to defines and execute rf runs using functions defined in the following scripts

# rf_models/
# ├── config.py          # run configs as dicts or dataclasses
# ├── spatial_cv.py      # spatial block splitting logic
# ├── train.py           # fit model given config + data, return results dict
# ├── evaluate.py        # metrics, plots
# ├── importance.py      # feature importance + permutation importance
# └── run.py             # orchestrates everything, saves outputs

from pathlib import Path
import json
import dataclasses
from pathlib import Path
import pandas as pd

from config import RunConfig
from spatial_cv import make_spatial_folds
from train import train_model
from evaluate import evaluate
from importance import get_feature_importance

In [30]:
##### Set-up (doesn't change)

# Get the current working directory
cd = Path.cwd().parent.parent 

# set directories
DATA_DIR = Path(f"{cd}/Data/Clean/Training_data/")
RESULTS_DIR = Path(f"{cd}/Results/RF_models/")
FOLD_DIR = f'/Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Fold_assignments/'

In [31]:
#### DEFINE RUN (MANUAL)

version = 'labor'
name_of_run =  'labor_rf_v1' 

target_var = 'log_labor_intensity_jobs_per_tonne' 
non_target_var = 'log_labor_intensity_jobs_per_million_USD'
# log_capital_intensity_USD_per_USD OR log_capital_intensity_USD_per_tonne
# log_labor_intensity_jobs_per_million_USD OR log_labor_intensity_jobs_per_tonne

type_of_model = 'rf' # rf or qrf

In [ ]:
##### Configuration (doesn't not change)

# config
fold = FOLD_DIR + version + '_folds.csv' 
model_data = f"{version}_final.csv"

RUNS = [
    RunConfig(
        run_name         = name_of_run,
        target           = target_var,   
        dataset          = model_data,
        fold_assignments = fold,
        model_type       = type_of_model,
        id_cols          = ["PROJ_ID", "country_ID", non_target_var],
        spatial_block_col = "country_ID",
    ),
]

In [32]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def save_results(results: dict, config: RunConfig, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)

    # config
    (out_dir / "config.json").write_text(
        json.dumps(dataclasses.asdict(config), indent=2)
    )

    # predictions across all folds
    results["predictions"].to_parquet(out_dir / "predictions.parquet", index=False)

    # per-fold best params
    params_df = pd.DataFrame([
        {"fold": r["fold"], **r["best_params"]}
        for r in results["fold_results"]
    ])
    params_df.to_csv(out_dir / "best_params.csv", index=False)

    # metrics and importance (already DataFrames)
    results["metrics"].to_csv(out_dir / "metrics.csv", index=False)
    results["importance"].to_csv(out_dir / "importance.csv", index=False)

    print(f"  saved to {out_dir}")


# ── Main ──────────────────────────────────────────────────────────────────────

def run(config: RunConfig):
    print(f"\n{'═'*60}")
    print(f"  run: {config.run_name}")
    print(f"  target: {config.target}  |  model: {config.model_type}")
    print(f"{'═'*60}")

    # load data
    df = pd.read_csv(DATA_DIR / config.dataset)

    # resolve feature cols dynamically
    feature_cols = [
        c for c in df.columns
        if c not in config.id_cols + [config.target]
    ]
    print(f"  features: {len(feature_cols)} columns")

    # spatial folds
    print("\n── Spatial folds ────────────────────────────────────────")
    folds = make_spatial_folds(df, config)

    # train
    print("\n── Training ─────────────────────────────────────────────")
    results = train_model(df, feature_cols, folds, config)

    # evaluate
    results["metrics"] = evaluate(results, config)

    # importance
    print("\n── Feature importance ───────────────────────────────────")
    results["importance"] = get_feature_importance(results, df, config)

    # save
    print("\n── Saving ───────────────────────────────────────────────")
    out_dir = RESULTS_DIR / config.run_name
    save_results(results, config, out_dir)


if __name__ == "__main__":
    for config in RUNS:
        run(config)


════════════════════════════════════════════════════════════
  run: labor_rf_v1
  target: log_labor_intensity_jobs_per_tonne  |  model: rf
════════════════════════════════════════════════════════════
  features: 36 columns

── Spatial folds ────────────────────────────────────────
  fold_1: 32 train countries (5,895 rows) | 1 test countries (5,136 rows)
  fold_2: 32 train countries (9,765 rows) | 1 test countries (1,266 rows)
  fold_3: 32 train countries (8,323 rows) | 1 test countries (2,708 rows)
  fold_4: 30 train countries (1,921 rows) | 3 test countries (9,110 rows)

── Training ─────────────────────────────────────────────

── fold_1 ──────────────────────────────
  test countries: ['BRA']
  best params: {'n_estimators': 500, 'min_samples_leaf': 2, 'max_features': 0.5, 'max_depth': 20}

── fold_2 ──────────────────────────────
  test countries: ['CAN']


/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


  best params: {'n_estimators': 500, 'min_samples_leaf': 2, 'max_features': 0.5, 'max_depth': 20}

── fold_3 ──────────────────────────────
  test countries: ['USA']
  best params: {'n_estimators': 200, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 20}

── fold_4 ──────────────────────────────
  test countries: ['BRA', 'CAN', 'USA']
  best params: {'n_estimators': 200, 'min_samples_leaf': 5, 'max_features': 0.3, 'max_depth': None}

── Evaluation ───────────────────────────────
   fold     rmse      mae       r2  rmse_orig  mae_orig  med_abs_err
 fold_1 1.275745 0.994416 0.441592   0.576906  0.063902     0.797056
 fold_2 1.165395 0.901263 0.418635   0.021741  0.005307     0.734181
 fold_3 1.053534 0.845636 0.374838   0.051239  0.006970     0.728882
 fold_4 1.771521 1.413461 0.106490   0.433001  0.041606     1.156463
overall 1.316549 1.038694 0.335389   0.270722  0.029446     0.854145

── Feature importance ───────────────────────────────────


/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



── Feature Importance (top 15) ──────────────────────────────
                                         feature  impurity_mean  impurity_std  permutation_mean  permutation_std
log_country_labor_intensity_jobs_per_million_USD       0.156879      0.115601          0.135959         0.094631
         probability_economic_land_use_objective       0.119270      0.037734          0.089206         0.041478
                                  crop_intensity       0.040572      0.009041          0.079493         0.045974
         probability_survival_land_use_objective       0.103849      0.016728          0.076396         0.069962
                sugar_crops_share_production_USD       0.042751      0.028041          0.064551         0.066473
                               share_large_field       0.043056      0.022874          0.053713         0.004708
                                             lon       0.070089      0.020561          0.052369         0.096279
                                 